# Економіка

Потік каменів духу і пігулок за ботами (tech 6.2, «потік валюти»). Камені — валюта (D17); джерела — події, чутки й здобич боїв (`loot:grp.*`), витрати — події, крамниця (`shop`), відновлення пулу (`qi_restore`) і поповнення місця (`place_refill`).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from simruns import BOTS, label, load, ranks

RUNS, SEED_FROM = 1000, 1  # per bot; data regenerates by itself when Core, Sim.Cli or content changes
data = load(runs=RUNS, seed_from=SEED_FROM)
runs = data.runs
pd.options.display.float_format = "{:.2f}".format
plt.rcParams["figure.figsize"] = (10, 4)


def named(df, axis=0):
    """Replaces content IDs in the index (axis=0) or columns (axis=1) with Ukrainian names."""
    return df.rename(index=label) if axis == 0 else df.rename(columns=label)


def per_run(long, value, by):
    """Mean of value per run for each bot, counting runs where it never happened as zero."""
    total = long.pivot_table(index=by, columns="bot", values=value, aggfunc="sum", fill_value=0)
    return total.div(runs.groupby("bot").size(), axis=1).reindex(columns=list(BOTS), fill_value=0)


## Камені за забіг

In [ ]:
cols = ["stones_start", "stones_earned", "stones_spent", "stones_end"]
runs.groupby("bot")[cols].agg(["mean", "median"]).reindex(list(BOTS))

## Джерела каменів

Середньо каменів на забіг за джерелом.

In [ ]:
earned = per_run(data.stones[data.stones.flow == "earned"], "amount", "source")
named(earned.sort_values(list(BOTS), ascending=False))

In [ ]:
named(earned).sort_values("cautious").plot.barh(figsize=(10, 5), title="Отримано каменів за забіг");

## Витрати каменів

In [ ]:
spent = per_run(data.stones[data.stones.flow == "spent"], "amount", "source")
named(spent.sort_values(list(BOTS), ascending=False))

In [ ]:
named(spent).sort_values("cautious").plot.barh(figsize=(10, 5), title="Витрачено каменів за забіг");

## Пігулки й фоліанти

Середньо куплено й прийнято за забіг (фоліанти лише купуються).

In [ ]:
pills = data.pills.pivot_table(index="item", columns=["flow", "bot"], values="count", aggfunc="sum", fill_value=0)
named(pills.div(runs.groupby("bot").size(), axis=1, level="bot"))

## Токсини

Піковий рівень токсинів за забіг і токсини на момент успішного прориву залежно від рангу.

In [ ]:
fig, ax = plt.subplots()
for bot in BOTS:
    ax.hist(runs.loc[runs.bot == bot, "peak_toxins"], bins=40, histtype="step", linewidth=1.5, label=bot)
ax.set(xlabel="пікові токсини", ylabel="забігів", title="Пікові токсини за забіг")
ax.legend();

In [ ]:
bt = data.breakthroughs
ok = bt[bt.result == "success"]
order = [r for r in ranks() if r in set(ok["rank"])]
ok.groupby(["bot", "rank"]).toxins.describe()[["count", "mean", "50%", "max"]].reindex(order, level="rank").rename(index=label)